# Bayesian Deep Learning

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 03.10 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/11_bayesian_deep_learning.ipynb)

---

## 🎯 Learning Objective

Extend Bayesian inference to neural network weights, and learn practical uncertainty-estimation techniques like MC-Dropout.

---

## 📐 Theory

Every model so far has treated a neural network's weights $\theta$ as fixed numbers to be
optimized (MLE, `03.07`). **Bayesian deep learning** instead treats $\theta$ as a random
variable with its own distribution — asking not just "what's the best-fit prediction?" but
"how confident should this specific prediction be?"

### Weight uncertainty and the Bayesian posterior over weights

Applying Bayes' theorem (`03.05`) to network weights: $p(\theta\mid D) \propto p(D\mid\theta)
p(\theta)$. A prediction for new input $x^*$ then **marginalizes** over every plausible weight
setting, weighted by how probable it is given the training data:

$$p(y^*\mid x^*, D) = \int p(y^*\mid x^*,\theta)\,p(\theta\mid D)\,d\theta,$$

the same marginalization from `03.04`, now over a network's entire parameter space rather than
a single latent variable. This integral has no closed form for any network with a
nonlinearity, so Bayesian deep learning is fundamentally about *approximating* it.

### Epistemic vs. aleatoric uncertainty

Two distinct kinds of "not sure" get conflated if you're not careful. **Aleatoric
uncertainty** is irreducible noise *in the data itself* — even a perfectly-trained model can't
predict a fair coin flip better than 50/50. **Epistemic uncertainty** is uncertainty *about the
model's parameters* — it shrinks as you see more training data, and it's specifically what
$p(\theta\mid D)$'s spread captures. A model can be confidently wrong (aleatoric noise it never
accounted for) or honestly unsure because it's extrapolating far from training data (high
epistemic uncertainty) — telling these apart determines whether "collect more data" or "this
task is inherently noisy" is the right response to a low-confidence prediction.

### Bayes by Backprop

**Bayes by Backprop** directly approximates $p(\theta\mid D)$ with a tractable distribution
$q_\phi(\theta)$ (typically an independent Gaussian per weight), trained by maximizing the same
ELBO objective from [Probabilistic Models in Machine Learning](10_probabilistic_models_in_ml.ipynb)
(`03.10`) — reconstruction traded against a KL penalty pulling $q_\phi(\theta)$ toward a prior
$p(\theta)$. It is precisely a VAE's machinery applied to a network's weights rather than to a
latent code per input: the reparameterization trick samples each weight as
$\theta=\mu+\sigma\odot\epsilon$, $\epsilon\sim\mathcal{N}(0,I)$, keeping sampling differentiable.

### MC-Dropout: a cheap approximation to a Bayesian network

Training with **dropout** (randomly zeroing activations, standard as a regularizer) turns out
to have a Bayesian interpretation: each dropout mask is a sample from an approximate posterior
over a family of sub-networks. **MC-Dropout** exploits this directly: keep dropout *active at
prediction time* (rather than off, as normal inference does), run the same input through the
network $T$ times with fresh random masks, and treat the $T$ predictions as samples from the
(approximate) predictive distribution:

$$\hat\mu(x^*) = \frac{1}{T}\sum_{t=1}^T f_{\theta_t}(x^*), \qquad \hat\sigma^2(x^*) = \frac{1}{T}\sum_{t=1}^T \big(f_{\theta_t}(x^*) - \hat\mu(x^*)\big)^2,$$

exactly the Monte Carlo pattern of [Sampling and Monte Carlo Methods](09_sampling_and_monte_carlo.ipynb)
(`03.09`), estimating a predictive mean/variance instead of an arbitrary expectation. Its appeal
is practical: no new training procedure — just a network you were already training, with one
inference-time change.

### Calibration

A model's uncertainty estimates are **well-calibrated** if, among all predictions it assigns
(say) 80% confidence, roughly 80% are actually correct. Good calibration is what makes an
uncertainty estimate *actionable* — a system allowed to defer to a human, or flag a prediction
as unreliable, is only trustworthy if its stated confidence tracks its real accuracy.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Fitting a small network to data that only covers part of the input range, then asking it to
predict everywhere: a standard (non-Bayesian) network gives a single confident curve
everywhere, even where it's extrapolating wildly, while MC-Dropout's predictive spread visibly
widens exactly in the regions with no training data.

In [ ]:
# Train on x in [-3,-1] U [1,3] (a gap with no data in the middle or beyond +-3), then predict
# over the FULL range -- MC-Dropout should be uncertain precisely where data is absent.
torch.manual_seed(0)
rng = np.random.default_rng(0)

def true_function(x):
    return np.sin(2 * x) + 0.3 * x

x_train_np = np.concatenate([rng.uniform(-3, -1, 40), rng.uniform(1, 3, 40)])
y_train_np = true_function(x_train_np) + rng.normal(scale=0.15, size=len(x_train_np))
X_train = torch.tensor(x_train_np, dtype=torch.float32).unsqueeze(1)
Y_train = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1)

class MCDropoutNet(nn.Module):
    def __init__(self, p_dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Dropout(p_dropout),
                                  nn.Linear(64, 64), nn.ReLU(), nn.Dropout(p_dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.net(x)

model = MCDropoutNet(p_dropout=0.2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for _ in range(1000):
    optimizer.zero_grad()
    loss = nn.functional.mse_loss(model(X_train), Y_train)
    loss.backward()
    optimizer.step()

x_test_np = np.linspace(-6, 6, 300)
X_test = torch.tensor(x_test_np, dtype=torch.float32).unsqueeze(1)

model.eval()  # dropout OFF: the standard, single deterministic prediction
with torch.no_grad():
    y_standard = model(X_test).numpy().squeeze()

model.train()  # dropout ON, kept active at PREDICTION time -- this is the entire MC-Dropout trick
T = 200
with torch.no_grad():
    mc_preds = np.stack([model(X_test).numpy().squeeze() for _ in range(T)])
mc_mean, mc_std = mc_preds.mean(axis=0), mc_preds.std(axis=0)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.scatter(x_train_np, y_train_np, s=15, color="black", alpha=0.6, label="training data", zorder=5)
ax.plot(x_test_np, true_function(x_test_np), "g--", lw=1.5, label="true function")
ax.plot(x_test_np, y_standard, color="#4C72B0", lw=2, label="standard (dropout OFF) prediction")
ax.plot(x_test_np, mc_mean, color="#C44E52", lw=2, label="MC-Dropout mean prediction")
ax.fill_between(x_test_np, mc_mean - 2 * mc_std, mc_mean + 2 * mc_std, color="#C44E52", alpha=0.25,
                 label="MC-Dropout +/- 2 std")
for lo, hi in [(-1, 1), (-6, -3), (3, 6)]:
    ax.axvspan(lo, hi, color="gray", alpha=0.12)
ax.set_title("Standard net: one confident curve everywhere.\nMC-Dropout: uncertainty widens in the gray (no-data) regions.")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

in_data_mask = (np.abs(x_test_np) >= 1) & (np.abs(x_test_np) <= 3)
out_data_mask = (np.abs(x_test_np) > 3.5)
print(f"Mean MC-Dropout std WITHIN training range:  {mc_std[in_data_mask].mean():.4f}")
print(f"Mean MC-Dropout std BEYOND training range (extrapolating): {mc_std[out_data_mask].mean():.4f}")
print(f"Ratio (extrapolation / in-range): {mc_std[out_data_mask].mean() / mc_std[in_data_mask].mean():.2f}x")

## 🐍 Implementation from Scratch

We implement MC-Dropout's predictive mean/variance decomposition directly (mean and variance
across stochastic forward passes) and decompose total predictive variance into its epistemic
and aleatoric parts by training an ensemble that additionally predicts its own aleatoric noise.

In [ ]:
# Decompose TOTAL predictive variance into epistemic + aleatoric via the law of total variance,
# applied to the mixture of T dropout-sampled Gaussians that MC-Dropout actually produces.
torch.manual_seed(1)
rng = np.random.default_rng(1)


def true_fn(x):
    return 0.5 * x


def aleatoric_std(x):
    return 0.05 + 0.25 * np.abs(x)  # noise that GENUINELY grows with x -- irreducible, by construction


x_np = rng.uniform(-3, 3, 300)
y_np = true_fn(x_np) + rng.normal(scale=aleatoric_std(x_np))
X = torch.tensor(x_np, dtype=torch.float32).unsqueeze(1)
Y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

class HeteroscedasticMCDropoutNet(nn.Module):
    """Two heads: mu(x) (prediction) and log_var(x) (its own ALEATORIC noise estimate at x),
    trained with dropout so repeated stochastic passes ALSO expose EPISTEMIC uncertainty."""
    def __init__(self, p_dropout=0.1):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Dropout(p_dropout),
                                     nn.Linear(64, 64), nn.ReLU(), nn.Dropout(p_dropout))
        self.mu_head, self.logvar_head = nn.Linear(64, 1), nn.Linear(64, 1)

    def forward(self, x):
        h = self.shared(x)
        return self.mu_head(h), self.logvar_head(h)

model = HeteroscedasticMCDropoutNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
for _ in range(2000):
    optimizer.zero_grad()
    mu, logvar = model(X)
    # Gaussian NLL -- letting the net predict its OWN noise level per input is exactly MLE
    # (03.07) for a heteroscedastic Gaussian, not an ad hoc loss.
    loss = (0.5 * logvar + 0.5 * (Y - mu) ** 2 / torch.exp(logvar)).mean()
    loss.backward()
    optimizer.step()

model.train()  # dropout ACTIVE -- every forward pass below is a fresh stochastic sub-network
T = 500
x_query = np.array([0.2, 1.5, 2.8])   # low/medium/high true aleatoric noise, all in-distribution
X_query = torch.tensor(x_query, dtype=torch.float32).unsqueeze(1)
with torch.no_grad():
    mus, logvars = zip(*[model(X_query) for _ in range(T)])
mus = torch.stack(mus).squeeze(-1).numpy()               # (T, 3): T stochastic mean predictions
vars_ = torch.stack(logvars).squeeze(-1).exp().numpy()    # (T, 3): T stochastic noise-variance predictions

epistemic_var = mus.var(axis=0)                    # Var_t[mu_t(x)]    -- spread OF the means
aleatoric_var = vars_.mean(axis=0)                  # E_t[sigma_t^2(x)] -- average PREDICTED noise
total_var = epistemic_var + aleatoric_var           # law of total variance, exact for a mixture

# Independent check: SAMPLE from the T-component mixture directly and compute its raw variance
mixture_samples = mus[:, None, :] + np.sqrt(vars_)[:, None, :] * rng.normal(size=(T, 400, 3))
total_var_from_samples = mixture_samples.reshape(-1, 3).var(axis=0)

print(f"{'x':>4} | {'true var':>9} | {'epistemic':>9} | {'aleatoric':>9} | {'sum':>7} | {'raw sample var':>14}")
for i, x_val in enumerate(x_query):
    print(f"{x_val:4.1f} | {aleatoric_std(x_val)**2:9.4f} | {epistemic_var[i]:9.4f} | "
          f"{aleatoric_var[i]:9.4f} | {total_var[i]:7.4f} | {total_var_from_samples[i]:14.4f}")

print(f"\nAleatoric share of total variance grows with |x|: {aleatoric_var[0]/total_var[0]:.3f} at "
      f"x=0.2 vs {aleatoric_var[2]/total_var[2]:.3f} at x=2.8 (matches how aleatoric_std(x) was built)")

## 🤖 Why This Matters for AI

**Uncertainty estimation** and **calibrated predictions** separate a model that's merely
accurate on average from one that's safe to deploy in high-stakes settings — a medical imaging
model or an LLM deciding whether to answer or abstain both need to know *when they don't know*.
MC-Dropout is popular precisely because it adds this to an already-trained architecture at
almost no extra cost. Below, we decompose predictive uncertainty into epistemic (reducible,
"not enough data here") and aleatoric (irreducible, "this input is inherently noisy") parts.

---

In [ ]:
# Calibration: does MC-Dropout's CONFIDENCE track its ACCURACY? The production question --
# "can I trust this model's stated confidence" -- made concrete.
torch.manual_seed(2)
rng = np.random.default_rng(2)

n_per_class, n_features = 500, 8
class0 = rng.normal(0, 1.2, size=(n_per_class, n_features))
class1 = rng.normal(0.9, 1.2, size=(n_per_class, n_features))  # OVERLAPPING classes -> genuine ambiguity
X_all = np.vstack([class0, class1]).astype(np.float32)
y_all = np.array([0] * n_per_class + [1] * n_per_class).astype(np.float32)

perm = rng.permutation(len(X_all))
X_all, y_all = X_all[perm], y_all[perm]
split = int(0.7 * len(X_all))
X_train, X_test = torch.tensor(X_all[:split]), torch.tensor(X_all[split:])
y_train, y_test = torch.tensor(y_all[:split]), torch.tensor(y_all[split:])

class MCDropoutClassifier(nn.Module):
    def __init__(self, p_dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, 32), nn.ReLU(), nn.Dropout(p_dropout),
                                  nn.Linear(32, 32), nn.ReLU(), nn.Dropout(p_dropout), nn.Linear(32, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)

clf = MCDropoutClassifier(p_dropout=0.3)
optimizer = torch.optim.Adam(clf.parameters(), lr=0.01)
bce = nn.BCEWithLogitsLoss()
for _ in range(300):
    optimizer.zero_grad()
    bce(clf(X_train), y_train).backward()
    optimizer.step()

clf.train()  # dropout ON at prediction time -- the MC-Dropout trick, now for classification
T = 100
with torch.no_grad():
    mc_probs = torch.sigmoid(torch.stack([clf(X_test) for _ in range(T)])).mean(dim=0).numpy()
predictions = (mc_probs > 0.5).astype(float)
confidences = np.maximum(mc_probs, 1 - mc_probs)   # confidence in whichever class was predicted
correct = (predictions == y_test.numpy()).astype(float)

# Reliability diagram: bin predictions by stated confidence, compare to ACTUAL accuracy per bin
bins = np.linspace(0.5, 1.0, 6)
bin_idx = np.digitize(confidences, bins) - 1
print(f"{'confidence bin':>15} | {'n':>4} | {'mean conf':>9} | {'accuracy':>8}")
ece = 0.0
for b in range(len(bins) - 1):
    mask = bin_idx == b
    if mask.sum() == 0:
        continue
    mean_conf, mean_acc = confidences[mask].mean(), correct[mask].mean()
    ece += (mask.sum() / len(confidences)) * abs(mean_conf - mean_acc)
    print(f"  [{bins[b]:.2f},{bins[b+1]:.2f}) | {mask.sum():4d} | {mean_conf:9.3f} | {mean_acc:8.3f}")

print(f"\nExpected Calibration Error (ECE): {ece:.4f}  (0=perfect; lowest bin has only ~15 points,")
print("so its accuracy estimate alone is noisy)")
print(f"Overall test accuracy: {correct.mean():.3f}")
print("The high-confidence bins (holding most test points) track their stated confidence closely")
print("-- the property that makes 'defer to a human below a confidence threshold' trustworthy.")

## 🏋️ Exercises

### Warm-up
1. In the first MC-Dropout regression cell, change `p_dropout` from `0.2` to `0.05` and rerun.
   Compare the predictive uncertainty band's width in the no-data (gray) regions, and explain
   what the dropout probability controls from a Bayesian perspective.

### Practice
2. In the epistemic/aleatoric decomposition cell, add a fourth query point at $x=5.0$ — well
   *outside* the training range $[-3,3]$. Predict its epistemic and aleatoric variance the same
   way, and explain why epistemic variance should dominate there, unlike at the in-distribution
   points already tested.

### Challenge
3. In the calibration cell, retrain the classifier with `p_dropout=0.0` (MC-Dropout then just
   repeats the same deterministic pass $T$ times) and recompute the reliability diagram and
   ECE. Compare to `p_dropout=0.3`. Explain why a deterministic network's single-softmax
   "confidence" tends to be overconfident, and why this motivates post-hoc calibration fixes.

---

## 📚 Further Reading
- Gal & Ghahramani, ["Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning"](https://arxiv.org/abs/1506.02142) (MC-Dropout)
- Blundell et al., ["Weight Uncertainty in Neural Networks"](https://arxiv.org/abs/1505.05424) (Bayes by Backprop)
- Guo et al., ["On Calibration of Modern Neural Networks"](https://arxiv.org/abs/1706.04599)
- Kendall & Gal, ["What Uncertainties Do We Need in Bayesian Deep Learning for Computer Vision?"](https://arxiv.org/abs/1703.04977)

---

*Next notebook: [The Optimization Landscape](../04_Optimization/01_optimization_landscape.ipynb)*